In [ ]:
from pathlib import Path
import duckdb
import numpy as np
import pandas as pd


In [90]:
PROJECT_ROOT = Path.cwd().parent
RAW_DATA = PROJECT_ROOT / "data" / "raw"
DB_FILE = PROJECT_ROOT / "nutrition.duckdb"



con = duckdb.connect(DB_FILE)

In [27]:
def load_csv(table_name):
    path = RAW_DATA / f"{table_name}.csv"

    con.execute(f"""
        CREATE TABLE {table_name} AS
        SELECT *
        FROM read_csv_auto(
            ?,
            sample_size=-1
        )
    """, [str(path)])

In [28]:
def table_exists(table_name):
    result = con.execute("""
        SELECT COUNT(*)
        FROM information_schema.tables
        WHERE table_name = ?
    """, [table_name]).fetchone()

    return result[0] > 0

In [30]:
required_tables = [
    "food",
    "nutrient",
    "food_nutrient",
    "food_portion",
    "measure_unit",
    "food_category",
    "branded_food"
]

for table in required_tables:
    if not table_exists(table):
        print(f"Loading {table}...")
        load_csv(table)
    else:
        print(f"{table} already exists.")

food already exists.
nutrient already exists.
food_nutrient already exists.
food_portion already exists.
measure_unit already exists.
Loading food_category...
Loading branded_food...


In [31]:
con.sql("DESCRIBE food").show()
con.sql("DESCRIBE nutrient").show()
con.sql("DESCRIBE food_nutrient").show()

┌──────────────────┬─────────────┬─────────┬─────────┬─────────┬─────────┐
│   column_name    │ column_type │  null   │   key   │ default │  extra  │
│     varchar      │   varchar   │ varchar │ varchar │ varchar │ varchar │
├──────────────────┼─────────────┼─────────┼─────────┼─────────┼─────────┤
│ fdc_id           │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ data_type        │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ description      │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ food_category_id │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ publication_date │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
└──────────────────┴─────────────┴─────────┴─────────┴─────────┴─────────┘

┌──────────────┬─────────────┬─────────┬─────────┬─────────┬─────────┐
│ column_name  │ column_type │  null   │   key   │ default │  extra  │
│   varchar    │   varchar   │ varchar │ varchar │ varchar │ varchar │
├──────────────┼─────────────┼──────

In [32]:
con.sql("SELECT * FROM food LIMIT 5").show()
con.sql("SELECT * FROM nutrient LIMIT 20").show()
con.sql("SELECT * FROM food_nutrient LIMIT 5").show()

┌─────────┬───────────────────┬───────────────────────────────────────────────────────────────────────────────────────────────────────────────────┬───────────────────────┬──────────────────┐
│ fdc_id  │     data_type     │                                                    description                                                    │   food_category_id    │ publication_date │
│  int64  │      varchar      │                                                      varchar                                                      │        varchar        │     varchar      │
├─────────┼───────────────────┼───────────────────────────────────────────────────────────────────────────────────────────────────────────────────┼───────────────────────┼──────────────────┤
│ 1105904 │ branded_food      │ WESSON Vegetable Oil 1 GAL                                                                                        │ Oils Edible           │ 2020-11-13       │
│ 1105905 │ branded_food      │ SWANSON BROTH

In [33]:
con.sql("""
SELECT
    f.fdc_id,
    f.description,
    f.data_type,
    n.name AS nutrient,
    n.unit_name,
    fn.amount
FROM food f
JOIN food_nutrient fn
    ON f.fdc_id = fn.fdc_id
JOIN nutrient n
    ON fn.nutrient_id = n.id
WHERE lower(f.description) LIKE '%cheddar%'
ORDER BY f.fdc_id, n.rank
LIMIT 100
""").show()

┌────────┬──────────────────────────────────────────────────────────┬────────────────┬─────────────────────────────┬───────────┬────────┐
│ fdc_id │                       description                        │   data_type    │          nutrient           │ unit_name │ amount │
│ int64  │                         varchar                          │    varchar     │           varchar           │  varchar  │ double │
├────────┼──────────────────────────────────────────────────────────┼────────────────┼─────────────────────────────┼───────────┼────────┤
│ 167964 │ Snacks, M&M MARS, COMBOS Snacks Cheddar Cheese Pretzel   │ sr_legacy_food │ Water                       │ G         │   1.67 │
│ 167964 │ Snacks, M&M MARS, COMBOS Snacks Cheddar Cheese Pretzel   │ sr_legacy_food │ Energy                      │ KCAL      │  463.0 │
│ 167964 │ Snacks, M&M MARS, COMBOS Snacks Cheddar Cheese Pretzel   │ sr_legacy_food │ Energy                      │ kJ        │ 1937.0 │
│ 167964 │ Snacks, M&M MARS, COMBO

In [34]:
def search_food(search_term, limit=20):
    return con.execute("""
        SELECT
            fdc_id,
            data_type,
            description,
            food_category_id
        FROM food
        WHERE lower(description) LIKE ?
        ORDER BY description
        LIMIT ?
    """, [f"%{search_term.lower()}%", limit]).fetchdf()

In [38]:
search_food("cheddar")

,fdc_id,data_type,description,food_category_id
0,2527760,branded_food,CHEDDAR & SOUR CREAM FLAVORED POTATO CHIPS,"Chips, Pretzels & Snacks"
1,2562699,branded_food,"CHEDDAR & SOUR CREAM FLAVORED POTATO CHIPS, C...","Chips, Pretzels & Snacks"
2,2588940,branded_food,EXTRA SHARP WHITE CHEDDAR CHEESE,Cheese
3,2597083,branded_food,EXTRA SHARP WHITE CHEDDAR CHEESE,Cheese
4,2703128,branded_food,Hillshire Farm Hearty Turkey & Cheddar Wedge,Sandwiches/Filled Rolls/Wraps
5,2718439,branded_food,Hillshire Farm Hearty Turkey & Cheddar Wedge,Sandwiches/Filled Rolls/Wraps
6,2731419,branded_food,Hillshire Farm Hearty Turkey & Cheddar Wedge,Sandwiches/Filled Rolls/Wraps
7,2734123,branded_food,Hillshire Farm Hearty Turkey & Cheddar Wedge,Sandwiches/Filled Rolls/Wraps
8,2739667,branded_food,Hillshire Farm Hearty Turkey & Cheddar Wedge,Sandwiches/Filled Rolls/Wraps
9,1640138,branded_food,"""""WHOLE CHEESE!"""" CRISPY BAKED CRACKERS, MILD ...",Flavored Snack Crackers


In [39]:
def get_nutrients(fdc_id):
    return con.execute("""
        SELECT
            n.id AS nutrient_id,
            n.name,
            n.unit_name,
            fn.amount
        FROM food_nutrient fn
        JOIN nutrient n
            ON fn.nutrient_id = n.id
        WHERE fn.fdc_id = ?
        ORDER BY n.rank
    """, [fdc_id]).fetchdf()

In [40]:
get_nutrients(2562699)

,nutrient_id,name,unit_name,amount
0,1008,Energy,KCAL,541.00
1,1003,Protein,G,7.06
2,1004,Total lipid (fat),G,35.29
3,1005,"Carbohydrate, by difference",G,54.12
4,1079,"Fiber, total dietary",G,4.70
5,2000,Total Sugars,G,4.71
6,1087,"Calcium, Ca",MG,47.00
7,1089,"Iron, Fe",MG,1.88
8,1092,"Potassium, K",MG,1200.00
9,1093,"Sodium, Na",MG,659.00


In [43]:
con.sql("""
        SELECT DISTINCT data_type
FROM food;
""").show()
con.sql("""
SELECT
    data_type,
    COUNT(*)
FROM food
GROUP BY data_type;
""").show()

┌──────────────────────────┐
│        data_type         │
│         varchar          │
├──────────────────────────┤
│ experimental_food        │
│ sr_legacy_food           │
│ agricultural_acquisition │
│ sample_food              │
│ branded_food             │
│ sub_sample_food          │
│ foundation_food          │
│ market_acquistion        │
│ survey_fndds_food        │
└──────────────────────────┘

┌──────────────────────────┬──────────────┐
│        data_type         │ count_star() │
│         varchar          │    int64     │
├──────────────────────────┼──────────────┤
│ sub_sample_food          │        75055 │
│ foundation_food          │          469 │
│ market_acquistion        │         7577 │
│ survey_fndds_food        │         5432 │
│ branded_food             │      1999950 │
│ sr_legacy_food           │         7793 │
│ agricultural_acquisition │          810 │
│ experimental_food        │          114 │
│ sample_food              │         4079 │
└────────────────────

In [44]:
con.sql("""
    SELECT *
FROM food
WHERE data_type='foundation_food'
LIMIT 10;
""").show()

con.sql("""
SELECT *
FROM food
WHERE data_type='sr_legacy_food'
LIMIT 10;
""").show()

┌────────┬─────────────────┬────────────────────────────────────────────────────────────────────────────┬──────────────────┬──────────────────┐
│ fdc_id │    data_type    │                                description                                 │ food_category_id │ publication_date │
│ int64  │     varchar     │                                  varchar                                   │     varchar      │     varchar      │
├────────┼─────────────────┼────────────────────────────────────────────────────────────────────────────┼──────────────────┼──────────────────┤
│ 321358 │ foundation_food │ Hummus, commercial                                                         │ 16               │ 2019-04-01       │
│ 321359 │ foundation_food │ Milk, reduced fat, fluid, 2% milkfat, with added vitamin A and vitamin D   │ 1                │ 2019-04-01       │
│ 321360 │ foundation_food │ Tomatoes, grape, raw                                                       │ 11               │ 2019-04-01 

In [45]:
con.sql("""
CREATE VIEW generic_foods AS
SELECT *
FROM food
WHERE data_type IN (
    'foundation_food',
    'sr_legacy_food',
    'survey_fndds_food'
);
""")

con.sql("""
CREATE VIEW branded_foods AS
SELECT *
FROM food
WHERE data_type='branded_food';
""")

In [46]:
con.sql("DESCRIBE food_portion").show()
con.sql("DESCRIBE measure_unit").show()

con.sql("SELECT * FROM food_portion LIMIT 10").show()
con.sql("SELECT * FROM measure_unit LIMIT 20").show()

┌─────────────────────┬─────────────┬─────────┬─────────┬─────────┬─────────┐
│     column_name     │ column_type │  null   │   key   │ default │  extra  │
│       varchar       │   varchar   │ varchar │ varchar │ varchar │ varchar │
├─────────────────────┼─────────────┼─────────┼─────────┼─────────┼─────────┤
│ id                  │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ fdc_id              │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ seq_num             │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ amount              │ DOUBLE      │ YES     │ NULL    │ NULL    │ NULL    │
│ measure_unit_id     │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ portion_description │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ modifier            │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ gram_weight         │ DOUBLE      │ YES     │ NULL    │ NULL    │ NULL    │
│ data_points         │ BIGINT      │ YES     │ NULL    │ NULL  

In [47]:
con.sql("""SELECT *
FROM measure_unit
WHERE id = 9999;""").show()

con.sql("""SELECT
    measure_unit_id,
    COUNT(*)
FROM food_portion
GROUP BY measure_unit_id
ORDER BY COUNT(*) DESC;""").show()

┌───────┬──────────────┐
│  id   │     name     │
│ int64 │   varchar    │
├───────┼──────────────┤
│  9999 │ undetermined │
└───────┴──────────────┘

┌─────────────────┬──────────────┐
│ measure_unit_id │ count_star() │
│      int64      │    int64     │
├─────────────────┼──────────────┤
│            9999 │        36545 │
│            1099 │         6129 │
│            1119 │          916 │
│            1000 │          585 │
│            1011 │          528 │
│            1010 │          395 │
│            1026 │          362 │
│            1058 │          330 │
│            1050 │          290 │
│            NULL │          273 │
│              ·  │            · │
│              ·  │            · │
│              ·  │            · │
│            1044 │           18 │
│            1069 │           14 │
│            1082 │           11 │
│            1038 │           11 │
│            1060 │            9 │
│            1009 │            8 │
│            1028 │            8 │
│        

In [48]:
# ---------------------------------------------------------
# ID SEQUENCES
# ---------------------------------------------------------

con.execute("""
CREATE SEQUENCE IF NOT EXISTS meal_id_seq START 1;
""")

con.execute("""
CREATE SEQUENCE IF NOT EXISTS log_id_seq START 1;
""")


# ---------------------------------------------------------
# NUTRIENT TARGETS
# nutrient_id comes from USDA nutrient.id
# ---------------------------------------------------------

con.execute("""
CREATE TABLE IF NOT EXISTS nutrient_targets (
    nutrient_id BIGINT PRIMARY KEY,
    target_amount DOUBLE NOT NULL,
    limit_type VARCHAR NOT NULL,      -- minimum, maximum, target
    period VARCHAR NOT NULL           -- daily, weekly
);
""")


# ---------------------------------------------------------
# FOOD PREFERENCES
# fdc_id comes from USDA food.fdc_id
# ---------------------------------------------------------

con.execute("""
CREATE TABLE IF NOT EXISTS food_preferences (
    fdc_id BIGINT PRIMARY KEY,
    preference VARCHAR NOT NULL,      -- preferred, acceptable, neutral, vetoed
    notes VARCHAR
);
""")


# ---------------------------------------------------------
# MEALS
# meal_id is generated automatically
# ---------------------------------------------------------

con.execute("""
CREATE TABLE IF NOT EXISTS meals (
    meal_id BIGINT PRIMARY KEY DEFAULT nextval('meal_id_seq'),
    meal_name VARCHAR NOT NULL
);
""")


# ---------------------------------------------------------
# MEAL INGREDIENTS
# meal_id references our meals table
# fdc_id references USDA food data
# ---------------------------------------------------------

con.execute("""
CREATE TABLE IF NOT EXISTS meal_ingredients (
    meal_id BIGINT NOT NULL,
    fdc_id BIGINT NOT NULL,
    grams DOUBLE NOT NULL,

    PRIMARY KEY (meal_id, fdc_id)
);
""")


# ---------------------------------------------------------
# WEEKLY PLAN
# Natural key is day + meal slot
# ---------------------------------------------------------

con.execute("""
CREATE TABLE IF NOT EXISTS weekly_plan (
    day_of_week INTEGER NOT NULL,
    meal_slot VARCHAR NOT NULL,
    meal_id BIGINT NOT NULL,

    PRIMARY KEY (day_of_week, meal_slot)
);
""")


# ---------------------------------------------------------
# FOOD LOG
# log_id is generated automatically
# ---------------------------------------------------------

con.execute("""
CREATE TABLE IF NOT EXISTS food_log (
    log_id BIGINT PRIMARY KEY DEFAULT nextval('log_id_seq'),
    eaten_at TIMESTAMP,
    input_text VARCHAR,

    fdc_id BIGINT,
    grams DOUBLE,

    match_confidence DOUBLE
);
""")


# ---------------------------------------------------------
# FOOD ALIASES
# Alias itself is the natural key
# ---------------------------------------------------------

con.execute("""
CREATE TABLE IF NOT EXISTS food_aliases (
    alias VARCHAR PRIMARY KEY,
    fdc_id BIGINT NOT NULL
);
""")

In [49]:
def calculate_food_nutrients(fdc_id, grams):
    return con.execute("""
        SELECT
            n.id AS nutrient_id,
            n.name,
            n.unit_name,
            fn.amount * (? / 100.0) AS amount_consumed
        FROM food_nutrient fn
        JOIN nutrient n
            ON fn.nutrient_id = n.id
        WHERE fn.fdc_id = ?
        ORDER BY n.rank
    """, [grams, fdc_id]).fetchdf()

In [50]:
calculate_food_nutrients(
    fdc_id=2588940,
    grams=50
)

,nutrient_id,name,unit_name,amount_consumed
0,1008,Energy,KCAL,196.500
1,1003,Protein,G,10.715
2,1004,Total lipid (fat),G,16.070
3,1005,"Carbohydrate, by difference",G,1.785
4,1079,"Fiber, total dietary",G,0.000
5,2000,Total Sugars,G,0.000
6,1087,"Calcium, Ca",MG,355.500
7,1089,"Iron, Fe",MG,0.000
8,1092,"Potassium, K",MG,0.000
9,1093,"Sodium, Na",MG,321.500


In [54]:
def get_portions(fdc_id):
    portions = con.execute("""
        SELECT
            fp.id AS portion_id,
            fp.amount,
            fp.modifier,
            fp.gram_weight,
            mu.name AS unit,
            'food_portion' AS source
        FROM food_portion fp
        LEFT JOIN measure_unit mu
            ON fp.measure_unit_id = mu.id
        WHERE fp.fdc_id = ?
        ORDER BY fp.gram_weight
    """, [fdc_id]).fetchdf()

    if not portions.empty:
        return portions

    branded = con.execute("""
        SELECT
            NULL AS portion_id,
            1.0 AS amount,
            household_serving_fulltext AS modifier,
            serving_size AS gram_weight,
            serving_size_unit AS unit,
            'branded_food' AS source
        FROM branded_food
        WHERE fdc_id = ?
    """, [fdc_id]).fetchdf()

    return branded

In [55]:
get_portions(2588940)

,portion_id,amount,modifier,gram_weight,unit,source
0,<NA>,1.0,1oz (28g),28.0,GRM,branded_food


In [ ]:
def calculate_portion_nutrients(fdc_id, portion_id=None, quantity=1):
    portions = get_portions(fdc_id)

    if portions.empty:
        raise ValueError(f"No portion information found for fdc_id {fdc_id}")

    if portion_id is not None:
        portion = portions[portions["portion_id"] == portion_id]

        if portion.empty:
            raise ValueError(
                f"Portion {portion_id} not found for fdc_id {fdc_id}"
            )

        portion = portion.iloc[0]

    else:
        # Default to the first available portion
        portion = portions.iloc[0]

    grams = portion["gram_weight"] * quantity

    return calculate_food_nutrients(
        fdc_id=fdc_id,
        grams=grams
    )



In [57]:
calculate_portion_nutrients(
    fdc_id=2588940,
    quantity=2
)

,nutrient_id,name,unit_name,amount_consumed
0,1008,Energy,KCAL,220.0800
1,1003,Protein,G,12.0008
2,1004,Total lipid (fat),G,17.9984
3,1005,"Carbohydrate, by difference",G,1.9992
4,1079,"Fiber, total dietary",G,0.0000
5,2000,Total Sugars,G,0.0000
6,1087,"Calcium, Ca",MG,398.1600
7,1089,"Iron, Fe",MG,0.0000
8,1092,"Potassium, K",MG,0.0000
9,1093,"Sodium, Na",MG,360.0800


In [58]:
def search_food(search_term, limit=20):
    return con.execute("""
        SELECT
            fdc_id,
            data_type,
            description,
            food_category_id
        FROM food
        WHERE lower(description) LIKE ?
        ORDER BY
            CASE data_type
                WHEN 'foundation_food' THEN 1
                WHEN 'sr_legacy_food' THEN 2
                WHEN 'survey_fndds_food' THEN 3
                WHEN 'branded_food' THEN 4
                ELSE 5
            END,
            length(description),
            description
        LIMIT ?
    """, [f"%{search_term.lower()}%", limit]).fetchdf()

In [59]:
search_food("waffle", 25)

,fdc_id,data_type,description,food_category_id
0,175039,sr_legacy_food,"Waffles, plain, prepared from recipe",18
1,175038,sr_legacy_food,"Waffles, plain, frozen, ready-to-heat",18
2,174628,sr_legacy_food,"Cereals ready-to-eat, Post, Waffle Crisp",8
3,167516,sr_legacy_food,"Waffles, buttermilk, frozen, ready-to-heat",18
4,174105,sr_legacy_food,"Waffles, gluten-free, frozen, ready-to-heat",18
5,168011,sr_legacy_food,"Van's, Gluten Free, Totally Original Waffles",18
6,167524,sr_legacy_food,"Waffles, chocolate chip, frozen, ready-to-heat",18
7,167519,sr_legacy_food,"Waffle, plain, frozen, ready-to-heat, microwave",18
8,175048,sr_legacy_food,"Waffles, plain, frozen, ready -to-heat, toasted",18
9,167517,sr_legacy_food,"Waffle, buttermilk, frozen, ready-to-heat, toa...",18


In [60]:
con.close()

In [80]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent
SRC_PATH = PROJECT_ROOT / "src"

if str(SRC_PATH) not in sys.path:
    sys.path.append(str(SRC_PATH))

from nutrition_assistant.repositories.food_repository import FoodRepository
from nutrition_assistant.engine.nutrition_engine import NutritionEngine

In [64]:
DB_PATH = PROJECT_ROOT / "database" / "nutrition.duckdb"

repo = FoodRepository(DB_PATH)
engine = NutritionEngine(repo)

In [69]:
display(repo.search("waffle", 10),
repo.get_portions(167516),
engine.calculate_portion_nutrients(
    fdc_id=167516,
    quantity=2
)

)

,fdc_id,data_type,description,food_category_id
0,175039,sr_legacy_food,"Waffles, plain, prepared from recipe",18
1,175038,sr_legacy_food,"Waffles, plain, frozen, ready-to-heat",18
2,174628,sr_legacy_food,"Cereals ready-to-eat, Post, Waffle Crisp",8
3,167516,sr_legacy_food,"Waffles, buttermilk, frozen, ready-to-heat",18
4,174105,sr_legacy_food,"Waffles, gluten-free, frozen, ready-to-heat",18
5,168011,sr_legacy_food,"Van's, Gluten Free, Totally Original Waffles",18
6,167524,sr_legacy_food,"Waffles, chocolate chip, frozen, ready-to-heat",18
7,167519,sr_legacy_food,"Waffle, plain, frozen, ready-to-heat, microwave",18
8,175048,sr_legacy_food,"Waffles, plain, frozen, ready -to-heat, toasted",18
9,167517,sr_legacy_food,"Waffle, buttermilk, frozen, ready-to-heat, toa...",18


,portion_id,amount,modifier,gram_weight,unit,source
0,81554,1.0,"waffle, round",38.0,undetermined,food_portion
1,81553,1.0,"waffle, square",39.0,undetermined,food_portion


,nutrient_id,name,unit_name,amount,amount_consumed
0,1051,Water,G,40.340,30.65840
1,1008,Energy,KCAL,273.000,207.48000
2,1062,Energy,kJ,1144.000,869.44000
3,1003,Protein,G,6.580,5.00080
4,1004,Total lipid (fat),G,9.220,7.00720
...,...,...,...,...,...
106,1226,Proline,G,0.559,0.42484
107,1227,Serine,G,0.347,0.26372
108,1018,"Alcohol, ethyl",G,0.000,0.00000
109,1057,Caffeine,MG,0.000,0.00000


In [75]:
from nutrition_assistant.repositories.meal_repository import MealRepository
import pandas as pd

meal_repo = MealRepository(repo.con)

pd.set_option("display.max_columns", None)   # Show all columns
pd.set_option("display.max_rows", None)      # Show all rows
pd.set_option("display.width", None)         # Don't constrain total width
pd.set_option("display.max_colwidth", None)  # Don't truncate cell contents


display(repo.search("Uncrustable", 10))

,fdc_id,data_type,description,food_category_id
0,981205,branded_food,"SMUCKER'S, UNCRUSTABLES, SPREAD SANDWICH, PEANUT BUTTER & RASPBERRY, PEANUT BUTTER & RASPBERRY",Frozen Appetizers & Hors D'oeuvres
1,1307639,branded_food,"SMUCKER'S, UNCRUSTABLES, SPREAD SANDWICH, PEANUT BUTTER & RASPBERRY, PEANUT BUTTER & RASPBERRY",Frozen Appetizers & Hors D'oeuvres
2,1627979,branded_food,"SMUCKER'S, UNCRUSTABLES, REDUCED SUGAR, PEANUT BUTTER & GRAPE SPREAD SANDWICH ON WHOLE WHEAT BREAD",Frozen Appetizers & Hors D'oeuvres
3,991771,branded_food,"SMUCKER'S, UNCRUSTABLES, REDUCED SUGAR, PEANUT BUTTER & GRAPE SPREAD SANDWICH ON WHOLE WHEAT BREAD",Frozen Appetizers & Hors D'oeuvres
4,1335828,branded_food,"SMUCKER'S, UNCRUSTABLES, REDUCED SUGAR, PEANUT BUTTER & GRAPE SPREAD SANDWICH ON WHOLE WHEAT BREAD",Frozen Appetizers & Hors D'oeuvres


In [86]:
from nutrition_assistant.models.ingredient import Ingredient
from nutrition_assistant.models.meal import Meal

lunch = Meal("PBJ Uncrustable")

lunch.add_ingredient(
    Ingredient(
        fdc_id=1307639,
        grams=78,
        name="PBJ Uncrustable"
    )
)

meal_id = meal_repo.create_meal(lunch)

print(meal_id)

meal_repo.list_meals()

3


,meal_id,meal_name,ingredient_count
0,2,PBJ Uncrustable,1
1,1,PBJ Uncrustable,1
2,3,PBJ Uncrustable,1


In [87]:
saved_lunch = meal_repo.get_meal(meal_id)

saved_lunch

Meal(name='PBJ Uncrustable', ingredients=[Ingredient(fdc_id=1307639, grams=78.0, name="SMUCKER'S, UNCRUSTABLES, SPREAD SANDWICH, PEANUT BUTTER & RASPBERRY, PEANUT BUTTER & RASPBERRY")])

In [88]:
import importlib
import nutrition_assistant.engine.nutrition_engine as nutrition_engine

importlib.reload(nutrition_engine)

NutritionEngine = nutrition_engine.NutritionEngine

engine = NutritionEngine(repo)

engine.calculate_meal(saved_lunch)

,nutrient_id,name,unit_name,amount_consumed
0,1003,Protein,G,9.4146
1,1004,Total lipid (fat),G,12.1056
2,1005,"Carbohydrate, by difference",G,33.6180
3,1008,Energy,KCAL,282.3600
4,1079,"Fiber, total dietary",G,4.0560
5,1087,"Calcium, Ca",MG,26.5200
6,1089,"Iron, Fe",MG,1.4508
7,1093,"Sodium, Na",MG,309.6600
8,1104,"Vitamin A, IU",IU,0.0000
9,1162,"Vitamin C, total ascorbic acid",MG,0.0000


In [91]:
con.execute("""
CREATE SEQUENCE IF NOT EXISTS target_profile_id_seq START 1;
""")

con.execute("""
CREATE TABLE IF NOT EXISTS target_profiles (
    profile_id BIGINT PRIMARY KEY DEFAULT nextval('target_profile_id_seq'),
    profile_name VARCHAR NOT NULL UNIQUE,
    notes VARCHAR
);
""")

con.execute("""
CREATE TABLE IF NOT EXISTS nutrient_targets_v2 (
    profile_id BIGINT NOT NULL,
    nutrient_id BIGINT NOT NULL,

    minimum_amount DOUBLE,
    target_amount DOUBLE,
    maximum_amount DOUBLE,

    reference_type VARCHAR,   -- RDA, AI, UL, AMDR, custom
    period VARCHAR NOT NULL DEFAULT 'daily',
    notes VARCHAR,

    PRIMARY KEY (profile_id, nutrient_id)
);
""")

con.execute("""
INSERT INTO target_profiles (profile_name, notes)
SELECT
    'default',
    'Baseline DRI-based nutrition targets'
WHERE NOT EXISTS (
    SELECT 1
    FROM target_profiles
    WHERE profile_name = 'default'
);
""")

In [96]:
con.close()

from nutrition_assistant.config import DATABASE_PATH

con = duckdb.connect(DATABASE_PATH)

print(DATABASE_PATH)
con.sql("SHOW TABLES").show()

C:\AI\Nutritional Assistant\database\nutrition.duckdb
┌──────────────────┐
│       name       │
│     varchar      │
├──────────────────┤
│ branded_food     │
│ branded_foods    │
│ food             │
│ food_aliases     │
│ food_category    │
│ food_log         │
│ food_nutrient    │
│ food_portion     │
│ food_preferences │
│ generic_foods    │
│ meal_ingredients │
│ meals            │
│ measure_unit     │
│ nutrient         │
│ nutrient_targets │
│ weekly_plan      │
└──────────────────┘
      16 rows     



In [97]:
target_nutrient_names = [
    "Protein",
    "Fiber, total dietary",
    "Calcium, Ca",
    "Iron, Fe",
    "Magnesium, Mg",
    "Potassium, K",
    "Sodium, Na",
    "Zinc, Zn",
    "Vitamin A, RAE",
    "Vitamin C, total ascorbic acid",
    "Vitamin D (D2 + D3)",
    "Vitamin E (alpha-tocopherol)",
    "Vitamin K (phylloquinone)",
    "Folate, total",
    "Vitamin B-12",
    "Fatty acids, total saturated",
    "Energy"
]

placeholders = ", ".join(["?"] * len(target_nutrient_names))

target_nutrients = con.execute(
    f"""
    SELECT
        id AS nutrient_id,
        name,
        unit_name
    FROM nutrient
    WHERE name IN ({placeholders})
    ORDER BY name
    """,
    target_nutrient_names
).fetchdf()

target_nutrients

,nutrient_id,name,unit_name
0,1087,"Calcium, Ca",MG
1,1008,Energy,KCAL
2,1062,Energy,kJ
3,1258,"Fatty acids, total saturated",G
4,1079,"Fiber, total dietary",G
5,1177,"Folate, total",UG
6,1089,"Iron, Fe",MG
7,1090,"Magnesium, Mg",MG
8,1092,"Potassium, K",MG
9,1003,Protein,G


In [98]:
found_names = set(target_nutrients["name"])
missing_names = set(target_nutrient_names) - found_names

missing_names

set()

In [100]:
target_nutrients = target_nutrients[
    ~(
        (target_nutrients["name"] == "Energy")
        & (target_nutrients["unit_name"] != "KCAL")
    )
].reset_index(drop=True)

target_nutrients

,nutrient_id,name,unit_name
0,1087,"Calcium, Ca",MG
1,1008,Energy,KCAL
2,1258,"Fatty acids, total saturated",G
3,1079,"Fiber, total dietary",G
4,1177,"Folate, total",UG
5,1089,"Iron, Fe",MG
6,1090,"Magnesium, Mg",MG
7,1092,"Potassium, K",MG
8,1003,Protein,G
9,1093,"Sodium, Na",MG


In [105]:
con.execute("""
CREATE SEQUENCE IF NOT EXISTS target_profile_id_seq START 1;
""")

con.execute("""
CREATE TABLE IF NOT EXISTS target_profiles (
    profile_id BIGINT PRIMARY KEY DEFAULT nextval('target_profile_id_seq'),
    profile_name VARCHAR NOT NULL UNIQUE,
    notes VARCHAR
);
""")

con.execute("""
CREATE TABLE IF NOT EXISTS nutrient_targets_v2 (
    profile_id BIGINT NOT NULL,
    nutrient_id BIGINT NOT NULL,

    minimum_amount DOUBLE,
    target_amount DOUBLE,
    maximum_amount DOUBLE,

    reference_type VARCHAR,
    period VARCHAR NOT NULL DEFAULT 'daily',
    notes VARCHAR,

    PRIMARY KEY (profile_id, nutrient_id)
);
""")

In [ ]:
con.execute("""
INSERT INTO target_profiles (profile_name, notes)
SELECT
    'default',
    'Baseline DRI-based nutrition targets'
WHERE NOT EXISTS (
    SELECT 1
    FROM target_profiles
    WHERE profile_name = 'default'
);
""")

In [106]:
profile_id = con.execute("""
SELECT profile_id
FROM target_profiles
WHERE profile_name = 'default'
""").fetchone()[0]


targets = [
    # nutrient_id, minimum, target, maximum, reference_type, notes

    (1008, None, 2576, None, "EER", "Estimated daily energy requirement"),
    (1003, 49, 49, None, "RDA", None),
    (1079, 36, 36, None, "AI", None),

    (1087, 1000, 1000, 2500, "RDA/UL", None),
    (1089, 8, 8, 45, "RDA/UL", None),
    (1090, 420, 420, None, "RDA", "Food magnesium has no established UL"),
    (1092, 3400, 3400, None, "AI", None),
    (1093, None, 1500, 2300, "AI/UL", None),
    (1095, 11, 11, 40, "RDA/UL", None),

    (1106, 900, 900, 3000, "RDA/UL", None),
    (1162, 90, 90, 2000, "RDA/UL", None),
    (1114, 15, 15, 100, "RDA/UL", None),
    (1109, 15, 15, 1000, "RDA/UL", None),
    (1185, 120, 120, None, "AI", None),
    (1177, 400, 400, 1000, "RDA/UL", None),
    (1178, 2.4, 2.4, None, "RDA", None),

    # Saturated fat has no DRI numerical target.
    # We'll add a custom planning constraint later.
]

TypeError: 'NoneType' object is not subscriptable

In [102]:
con.sql("SHOW TABLES").show()

┌──────────────────┐
│       name       │
│     varchar      │
├──────────────────┤
│ branded_food     │
│ branded_foods    │
│ food             │
│ food_aliases     │
│ food_category    │
│ food_log         │
│ food_nutrient    │
│ food_portion     │
│ food_preferences │
│ generic_foods    │
│ meal_ingredients │
│ meals            │
│ measure_unit     │
│ nutrient         │
│ nutrient_targets │
│ weekly_plan      │
└──────────────────┘
      16 rows     

